## Imports y conexión

In [3]:
import os
from pathlib import Path
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.width', 200)

load_dotenv()

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME_ORIGEN")

url = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(url)

OUT_DIR = Path("../docs/metadata_origen")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Helper para que pandas + SQLAlchemy 2.x funcione bien
def run_query(sql: str) -> pd.DataFrame:
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)

print("Conexión OK ✓")

Conexión OK ✓


## Inventario de tablas con conteo de filas

In [4]:
sql_tables = """
SELECT 
    table_name,
    (xpath('/row/cnt/text()', xml_count))[1]::text::int AS row_count
FROM (
    SELECT 
        table_name,
        query_to_xml(format('SELECT COUNT(*) AS cnt FROM public.%I', table_name), 
                     false, true, '') AS xml_count
    FROM information_schema.tables
    WHERE table_schema = 'public' AND table_type = 'BASE TABLE'
) t
ORDER BY row_count DESC;
"""
df_tables = run_query(sql_tables)
df_tables.to_csv(OUT_DIR / "01_tablas.csv", index=False)
df_tables

,table_name,row_count
0,sale_item,42555
1,sale,20000
2,customer,5750
3,return_item,2330
4,inventory,1000
5,product,50
6,central_product,49
7,central_inventory,49
8,city_zone,42
9,warehouse_location,40


## Columnas, tipos y nullabilidad

In [5]:
sql_columns = """
SELECT 
    table_name,
    ordinal_position AS pos,
    column_name,
    data_type,
    character_maximum_length AS max_len,
    numeric_precision AS num_prec,
    numeric_scale AS num_scale,
    is_nullable,
    column_default
FROM information_schema.columns
WHERE table_schema = 'public'
ORDER BY table_name, ordinal_position;
"""
df_columns = run_query(sql_columns)
df_columns.to_csv(OUT_DIR / "02_columnas.csv", index=False)
print(f"Total columnas: {len(df_columns)}")
df_columns

Total columnas: 96


,table_name,pos,column_name,data_type,max_len,num_prec,num_scale,is_nullable,column_default
0,brand,1,brand_id,integer,NaN,32.0,0.0,NO,nextval('brand_brand_id_seq'::regclass)
1,brand,2,name,character varying,150.0,NaN,NaN,NO,NaN
2,brand,3,country,character varying,100.0,NaN,NaN,YES,NaN
3,brand,4,website,character varying,200.0,NaN,NaN,YES,NaN
4,category,1,category_id,integer,NaN,32.0,0.0,NO,nextval('category_category_id_seq'::regclass)
...,...,...,...,...,...,...,...,...,...
91,warehouse_location,2,warehouse_id,integer,NaN,32.0,0.0,YES,NaN
92,warehouse_location,3,zone,character varying,50.0,NaN,NaN,YES,NaN
93,warehouse_location,4,aisle,character varying,10.0,NaN,NaN,YES,NaN
94,warehouse_location,5,shelf,character varying,10.0,NaN,NaN,YES,NaN


## Claves primarias

In [6]:
sql_pks = """
SELECT 
    tc.table_name,
    kcu.column_name,
    kcu.ordinal_position AS pk_pos,
    tc.constraint_name
FROM information_schema.table_constraints tc
JOIN information_schema.key_column_usage kcu 
    ON tc.constraint_name = kcu.constraint_name
    AND tc.table_schema = kcu.table_schema
WHERE tc.constraint_type = 'PRIMARY KEY' 
    AND tc.table_schema = 'public'
ORDER BY tc.table_name, kcu.ordinal_position;
"""
df_pks = run_query(sql_pks)
df_pks.to_csv(OUT_DIR / "03_primary_keys.csv", index=False)
print(f"Tablas con PK: {df_pks['table_name'].nunique()} de {len(df_tables)}")
df_pks

Tablas con PK: 17 de 17


,table_name,column_name,pk_pos,constraint_name
0,brand,brand_id,1,brand_pkey
1,category,category_id,1,category_pkey
2,central_inventory,inventory_id,1,central_inventory_pkey
3,central_product,product_id,1,central_product_pkey
4,city_zone,postal_code,1,city_zone_pkey
5,customer,customer_id,1,customer_pkey1
6,inventory,inventory_id,1,inventory_pkey
7,offer,offer_id,1,offer_pkey
8,product,product_id,1,product_pkey
9,product_offer,product_id,1,product_offer_pkey


## Claves foráneas (relaciones)

In [7]:
sql_fks = """
SELECT
    tc.table_name AS tabla_origen,
    kcu.column_name AS col_origen,
    ccu.table_name AS tabla_destino,
    ccu.column_name AS col_destino,
    tc.constraint_name
FROM information_schema.table_constraints AS tc
JOIN information_schema.key_column_usage AS kcu
    ON tc.constraint_name = kcu.constraint_name
    AND tc.table_schema = kcu.table_schema
JOIN information_schema.constraint_column_usage AS ccu
    ON ccu.constraint_name = tc.constraint_name
    AND ccu.table_schema = tc.table_schema
WHERE tc.constraint_type = 'FOREIGN KEY'
    AND tc.table_schema = 'public'
ORDER BY tc.table_name, kcu.ordinal_position;
"""
df_fks = run_query(sql_fks)
df_fks.to_csv(OUT_DIR / "04_foreign_keys.csv", index=False)
print(f"Total FKs: {len(df_fks)}")
df_fks

Total FKs: 16


,tabla_origen,col_origen,tabla_destino,col_destino,constraint_name
0,central_inventory,product_id,central_product,product_id,central_inventory_product_id_fkey
1,central_inventory,location_id,warehouse_location,location_id,central_inventory_location_id_fkey
2,central_inventory,warehouse_id,warehouse,warehouse_id,central_inventory_warehouse_id_fkey
3,central_product,brand_id,brand,brand_id,central_product_brand_id_fkey
4,central_product,category_id,category,category_id,central_product_category_id_fkey
5,inventory,product_id,product,product_id,inventory_product_id_fkey
6,inventory,store_id,store,store_id,inventory_store_id_fkey
7,product_offer,product_id,product,product_id,product_offer_product_id_fkey
8,product_offer,offer_id,offer,offer_id,product_offer_offer_id_fkey
9,return_item,sale_item_id,sale_item,sale_item_id,return_item_sale_item_id_fkey


## Tablas "huérfanas" (sin relaciones FK ni entrantes ni salientes)

In [8]:
tablas_con_fk_saliente = set(df_fks['tabla_origen'].unique())
tablas_con_fk_entrante = set(df_fks['tabla_destino'].unique())
todas_tablas = set(df_tables['table_name'].unique())

huerfanas = todas_tablas - tablas_con_fk_saliente - tablas_con_fk_entrante

print("Tablas SIN ninguna relación FK declarada:")
for t in sorted(huerfanas):
    print(f"  - {t}")

Tablas SIN ninguna relación FK declarada:
  - city_zone
  - return_reason


## Vista resumen por tabla (todo junto)

In [9]:
def resumen_tabla(nombre_tabla):
    """Imprime un resumen completo de una tabla."""
    print(f"\n{'='*70}")
    print(f"TABLA: {nombre_tabla}")
    print(f"{'='*70}")
    
    # Filas
    filas = df_tables[df_tables['table_name'] == nombre_tabla]['row_count'].values
    print(f"Filas: {filas[0] if len(filas) else 'N/A'}")
    
    # Columnas
    cols = df_columns[df_columns['table_name'] == nombre_tabla][
        ['pos', 'column_name', 'data_type', 'is_nullable', 'column_default']
    ]
    print(f"\nColumnas ({len(cols)}):")
    print(cols.to_string(index=False))
    
    # PK
    pk = df_pks[df_pks['table_name'] == nombre_tabla]['column_name'].tolist()
    print(f"\nPK: {pk if pk else '⚠️ SIN PK'}")
    
    # FKs salientes
    fks_out = df_fks[df_fks['tabla_origen'] == nombre_tabla]
    if len(fks_out):
        print(f"\nFKs salientes (apunta a otras tablas):")
        for _, fk in fks_out.iterrows():
            print(f"  {fk['col_origen']} → {fk['tabla_destino']}.{fk['col_destino']}")
    
    # FKs entrantes
    fks_in = df_fks[df_fks['tabla_destino'] == nombre_tabla]
    if len(fks_in):
        print(f"\nFKs entrantes (otras tablas la referencian):")
        for _, fk in fks_in.iterrows():
            print(f"  {fk['tabla_origen']}.{fk['col_origen']} → {fk['col_destino']}")

# Imprime resumen de todas las tablas (de mayor a menor volumen)
for t in df_tables['table_name']:
    resumen_tabla(t)


TABLA: sale_item
Filas: 42555

Columnas (7):
 pos  column_name data_type is_nullable                                  column_default
   1 sale_item_id   integer          NO nextval('sale_item_sale_item_id_seq'::regclass)
   2      sale_id   integer         YES                                             NaN
   3   product_id   integer         YES                                             NaN
   4     quantity   integer          NO                                             NaN
   5   unit_price   numeric          NO                                             NaN
   6     offer_id   integer         YES                                             NaN
   7     subtotal   numeric          NO                                             NaN

PK: ['sale_item_id']

FKs salientes (apunta a otras tablas):
  product_id → product.product_id
  offer_id → offer.offer_id
  sale_id → sale.sale_id

FKs entrantes (otras tablas la referencian):
  return_item.sale_item_id → sale_item_id

TABLA: sale
